# NLP 课堂实践：构建React Agent系统

**实践内容概述**: 部署React Agent系统, 首先完成一个简单联网查询的小例子，掌握React的基本迭代思想。然后以Hothop QA作为实验数据集, 进一步探究React Agent的核心思想

**实践目录**：
1. **React基础知识**：掌握React Agent基本思想和结构
2. **React Agent环境配置**：掌握React Agent环境部署、配置大模型API、动手实践基本的react联网查询应用。在此基础上，下载Hothop数据集来 QA和EM、F1指标来评测React Agent能力
3. **实验测试**：编写实验代码，完成React Agent构建及评估框架
4. **量化评估**：根据实验结果，探究直接调用LLM 和 利用React Agent完成QA的效果
5. **课后作业**：通过三道课后作业，自行设计新的tool工具，进一步加深对Agent理解

## React Agent 基础知识
Agent作为能够感知环境、做出决策并采取行动以实现特定目标的自主实体。与传统直接调用语言模型相比，Agent 具备以下核心特征：
* 自主性：无需人工干预即可独立运行
* 反应性：能对环境变化做出实时响应
* 主动性：主动追求目标而非被动响应
* 社会性：像人类进行交互

React Agent = Reasoning + Acting， 即不让LLM一次性回答出问题的最终答案，而是让它不断地与外界真实环境和工具进行交互，通过多次“思考”和“行动”，根据环境反馈，再进行下一步决策

React每一轮迭代LLM输出结构：

(1) Thought: 思考为什么要这么做（LLM 内部推理）

(2) Action: 调用哪些 Tool（函数名，例如 "search"）

(3) Observation: 工具返回运行结果

(4) Loop： Agent根据观察到的结果继续思考
 

ReAct是一个常用的基础框架，相对于Plan-and-Solve等Agent框架，其复杂度低，控制性强
![图片说明](image.png)



## Agent环境配置
1.配置python虚拟环境
```python
conda create --name react  python==3.11 -y
conda activate react
```
2.克隆项目到本地
```python
git clone https://github.com/EthantLu/code-demo-agent.git
cd DM-Code-Agent
```
3.安装环境
```python
pip install --upgrade pip
pip install -e ".[dev]"
```
4.配置LLM API
```python
cp .env.example .env
OPENAI_API_KEY=sk-你的中转key
OPENAI_BASE_URL=https://你的中转域名.com/v1
OPENAI_MODEL=gpt-4o-mini
```

## 实验测试
1. 检查模块及环境配置有效性
```python
from __future__ import annotations
import subprocess
import sys
import dm_agent
print("dm_agent OK")

out = subprocess.run(
    ["dm-agent-bench", "--help"], capture_output=True, text=True
).stdout
print(out[:400])

```

2.下载 SWE-bench Lite 数据集
本次课堂实践使用固定含有50条测试条目的数据集子集
```python   
from dm_agent.benchmarks.swebench_lite.loader import (
    _row_to_instance,
    _snapshot_path,
    snapshot_to_jsonl,
    load_instances,
    fixed_subset_50,
)

snapshot = _snapshot_path()
if snapshot.exists():
    print(f"[cache] snapshot already exists: {snapshot}")
else:
    print(f"[modelscope] snapshot 不存在，准备从 ModelScope 下载 ...")
    try:
        from modelscope.hub.snapshot_download import snapshot_download
    except ImportError as exc:
        print(f"ERROR: 导入 modelscope 失败: {exc!r}")
        print("如果是 'No module named modelscope' → pip install modelscope")
        sys.exit(1)
    try:
        import pyarrow.parquet as pq
    except ImportError:
        print("ERROR: 缺少 pyarrow，请先安装：\n    pip install pyarrow")
        sys.exit(1)

    try:
        repo_dir = snapshot_download(
            "princeton-nlp/SWE-bench_Lite",
            repo_type="dataset",
        )
    except Exception as exc:
        print(f"ERROR: ModelScope 拉取失败: {exc}")
        sys.exit(1)

    from pathlib import Path as _Path
    parquet_files = sorted(_Path(repo_dir).rglob("*.parquet"))
    if not parquet_files:
        print(f"ERROR: 在 {repo_dir} 下没找到 parquet 文件")
        sys.exit(1)
    print(f"[modelscope] parquet files: {[p.name for p in parquet_files]}")

    rows: list[dict] = []
    for pf in parquet_files:
        table = pq.read_table(pf)
        rows.extend(table.to_pylist())
    print(f"[modelscope] 共读取 {len(rows)} 条 row")

    instances = [_row_to_instance(row) for row in rows]
    snapshot.parent.mkdir(parents=True, exist_ok=True)
    snapshot_to_jsonl(instances, snapshot)
    print(f"[modelscope] snapshot written: {snapshot}  ({len(instances)} instances)")


all_instances = load_instances()
print(f"全量 SWE-bench Lite: {len(all_instances)} instances")

subset = fixed_subset_50(instances=all_instances)
print(f"固定子集: {len(subset)} instances\n")
# problem_statement：告诉模型要解决什么问题。
# base_commit：告诉评测系统从哪个代码版本开始。
# patch：官方正确答案补丁。
# fail_to_pass / pass_to_pass：告诉评测系统哪些测试用于判定修复是否成功、是否引入回归。
print("\n" + "=" * 80)
print("前 3 条完整测试数据")
print("=" * 80)
from dataclasses import asdict
import json as _json
for i, inst in enumerate(subset[:3], 1):
    print(f"\n--- [{i}] {inst.instance_id} ---")
    print(_json.dumps(asdict(inst), indent=2, ensure_ascii=False))
```
3. 配置大语言模型API
```shell
cp .env .env.example
```
```shell
OPENAI_API_KEY=
OPENAI_BASE_URL=
OPENAI_MODEL=
```